<a href="https://colab.research.google.com/github/majavier26/DSProjects/blob/main/Anomaly%20detection/Fraud_Detection_of_Bank_Transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install kaggle
! pip install geocoder
! pip install ipinfo

In [2]:
from google.colab import drive
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

# Location
from geopy.geocoders import Nominatim
import geocoder
import ipinfo
from geopy import distance


# Machine learning
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.cluster import DBSCAN

# Neural networks
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten
from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


# Fraud Detection of Bank Transactions

We are using this [Kaggle dataset](https://www.kaggle.com/datasets/valakhorasani/bank-transaction-dataset-for-fraud-detection). We will be employing two different techniques for detecting fraudulent transactions: a clustering technique called `DBSCAN` and a convolutional neural network called `AutoEncoder`.

## Preprocessing data

### Obtaining data



In [4]:
# Check for my Kaggle API
if os.path.exists('~/.kaggle/') != True:
  ! mkdir ~/.kaggle/
  ! cp /content/drive/MyDrive/Kaggle_API_Credentials/kaggle.json ~/.kaggle/
  ! chmod 600 ~/.kaggle/kaggle.json
# Check if I already have the data
if os.path.exists('/content/bank-transaction-dataset-for-fraud-detection.zip') != True:
  ! kaggle datasets download valakhorasani/bank-transaction-dataset-for-fraud-detection
  ! unzip bank-transaction-dataset-for-fraud-detection.zip

Dataset URL: https://www.kaggle.com/datasets/valakhorasani/bank-transaction-dataset-for-fraud-detection
License(s): apache-2.0
Archive:  bank-transaction-dataset-for-fraud-detection.zip
  inflating: bank_transactions_data_2.csv  


### Reading data

In [5]:
data = pd.read_csv('/content/bank-transaction-dataset-for-fraud-detection.zip')
data

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2507,TX002508,AC00297,856.21,2023-04-26 17:09:36,Credit,Colorado Springs,D000625,21.157.41.17,M072,Branch,33,Doctor,109,1,12690.79,2024-11-04 08:11:29
2508,TX002509,AC00322,251.54,2023-03-22 17:36:48,Debit,Tucson,D000410,49.174.157.140,M029,Branch,48,Doctor,177,1,254.75,2024-11-04 08:11:42
2509,TX002510,AC00095,28.63,2023-08-21 17:08:50,Debit,San Diego,D000095,58.1.27.124,M087,Branch,56,Retired,146,1,3382.91,2024-11-04 08:08:39
2510,TX002511,AC00118,185.97,2023-02-24 16:24:46,Debit,Denver,D000634,21.190.11.223,M041,Online,23,Student,19,1,1776.91,2024-11-04 08:12:22


Let's check the data if there are NaN values.

In [6]:
data.describe()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance
count,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000
mean,297.593778,44.673965,119.643312,1.124602,5114.302966
std,291.946243,17.792198,69.963757,0.602662,3900.942499
min,0.260000,18.000000,10.000000,1.000000,101.250000
25%,81.885000,27.000000,63.000000,1.000000,1504.370000
50%,211.140000,45.000000,112.500000,1.000000,4735.510000
75%,414.527500,59.000000,161.000000,1.000000,7678.820000
max,1919.110000,80.000000,300.000000,5.000000,14977.990000


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   TransactionID            2512 non-null   object 
 1   AccountID                2512 non-null   object 
 2   TransactionAmount        2512 non-null   float64
 3   TransactionDate          2512 non-null   object 
 4   TransactionType          2512 non-null   object 
 5   Location                 2512 non-null   object 
 6   DeviceID                 2512 non-null   object 
 7   IP Address               2512 non-null   object 
 8   MerchantID               2512 non-null   object 
 9   Channel                  2512 non-null   object 
 10  CustomerAge              2512 non-null   int64  
 11  CustomerOccupation       2512 non-null   object 
 12  TransactionDuration      2512 non-null   int64  
 13  LoginAttempts            2512 non-null   int64  
 14  AccountBalance          

There are zero NaN values in the data.

### Transforming the columns

**Columns to be removed**

- `TransactionID`, `AccountID`
- At first glance, we could also remove `DeviceID` and `MerchantID` because these are just identification of the client, however certain devices like burner phones are more likely to be used for fraudulent transactions. Criminals could also work with the bank agents, thus `MerchantID` is also important.
- I also wanted to remove `IP Address` as I thought it would be irrelevant, but it actually reveals the location of the transaction.

**Numerical variables**
- `TransactionAmount`
- `CustomerAge`
- `TransactionDuration`
- `LoginAttempts`
- `AccountBalance`
- Treatment: `StandardScaler`

**Cardinal variables**
- `TransactionType`
- `Channel`
- `CustomerOccupation`
- Treatment: `OneHotEncoder`
- `Location`
- `IP Address`
- Treatment 1: Turn the IP address into a `GeoLocation` and we can either see if the `Location` and the `GeoLocation` belong in the same country, giving us 1 in the `isSameCountry` column, and 0 if not in the same country.
- Treatment 2: Turn the IP address into a `GeoLocation` and we can get the distance between the `Location` and `GeoLocation`. If it's under 100 km, we label the column `isClose` with 1, else 0.

**Datetime variables**
- `TransactionDate`
- `PreviousTransactionDate`
- Treatment: `TimeDelta` of the two columns

We can separate the numerical columns from the categorical variables. All of our categorical variables are cardinal, and nothing is ordinal or boolean thus we can call all of our cardinal variables as categorical.

In [8]:
num_column = data[['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']]
cat_column = data[['TransactionType', 'Channel', 'CustomerOccupation']]

#### Numerical variables

As usual, we will use scale the numerical variables using `MinMaxScaler`. Even though there are no missing values, let's still impute missing values with the mean for replicability.

In [9]:
num_processor = Pipeline(
    steps=[
           ('imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
           ('scaler', MinMaxScaler())
    ]
)

In [10]:
num_processor

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', MinMaxScaler())])

##### Datetime variables

The two datetimes `TransactionDate` and `PreviousTransactionDate` don't really tell us what is going on. What matters is the interval between the two, as people don't really make bank transactions frequently. Fraudulent transactions on the other hand may have a small interval of transaction time as:
- the fraud might be accessing the victim's account after they have just accessed theirs, as they may have captured their password
- the fraud might be practicing or trying to crack the account thus multiple login attemps have been made and the interval between the two transactions is short

In [11]:
# Convert the columns to a Pandas datetime object, subtract them, and convert the timedelta into seconds
interval_transaction_sec = (pd.to_datetime(data['PreviousTransactionDate']) - pd.to_datetime(data['TransactionDate'])).dt.total_seconds()
interval_transaction_sec

,0
0,49477134.0
1,42823516.0
2,41694656.0
3,47403415.0
4,33228915.0
...,...
2507,48178913.0
2508,51201294.0
2509,38069989.0
2510,53452056.0


And let's put this into our `num_column`.

In [12]:
num_column['interval'] = interval_transaction_sec
num_column

<ipython-input-12-afe9b9d38787>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  num_column['interval'] = interval_transaction_sec


,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,interval
0,14.09,70,81,1,5112.21,49477134.0
1,376.24,68,141,1,13758.91,42823516.0
2,126.29,19,56,1,1122.35,41694656.0
3,184.50,26,25,1,8569.06,47403415.0
4,13.45,26,198,1,7429.40,33228915.0
...,...,...,...,...,...,...
2507,856.21,33,109,1,12690.79,48178913.0
2508,251.54,48,177,1,254.75,51201294.0
2509,28.63,56,146,1,3382.91,38069989.0
2510,185.97,23,19,1,1776.91,53452056.0


#### Categorical variables

The categorical variables with their dictionary are as follows:

- `TransactionType` (Debit, Card)
- `Channel` (ATM, Online, Branch)
- `CustomerOccupation` (Student, Engineer, Doctor, Retired)

Since we already know what

##### OneHotEncoding

In [13]:
card_processor = Pipeline(
    steps=[
        ('imputer', SimpleImputer(fill_value='missing', strategy='constant')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [14]:
card_processor

Pipeline(steps=[('imputer',
                 SimpleImputer(fill_value='missing', strategy='constant')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])

##### Location variables

We will go with the easier route and implement the `isSameCountry` column.

**IP Address column**

In [15]:
ip_address_df = data['IP Address']

In [16]:
ip0 = ip_address_df[10]
ip0

'213.15.9.253'

When we get the country of the IP address, it only seems to output the country code.

In [17]:
geocoder.ip(ip0).country

'IN'

With 'IN' being India. We should also get the country code for the location column too.

Doing this for all of the IP addresses:

###### IP Info

In [63]:
ipFinder = ipinfo.getHandler('75742cfebed550')
country = ipFinder.getDetails(ip0).country
country

'IN'

In [83]:
# ip_country_codes = []
# ip_counter = 0

# for ip_address in ip_address_df:
#   try:
#     country_code = ipFinder.getDetails(ip_address).country
#     ip_country_codes.append(country_code)
#     ip_counter += 1
#   except AttributeError:
#     ip_country_codes.append(None)
#     ip_counter += 1

###### Geocoder

In [ ]:
# Generating country codes and appending
ip_country_codes = []
counter = 0
for ip_address in ip_address_df:
  # For loop rests so the geocoder doesn't get overwhelmed
  if counter % 100 == 0:
    print('Sleeping for a minute...')
    time.sleep(70)
    country_code = geocoder.ip(ip_address).country
    ip_country_codes.append(country_code)
    counter += 1
  else:
    country_code = geocoder.ip(ip_address).country
    ip_country_codes.append(country_code)
    counter += 1

# Tacking ip codes to the original dataframe
ip_address_df['IP_Code'] = ip_country_codes

In [35]:
# Generating country codes and appending
ip_country_codes = []
counter = 0
for ip_address in ip_address_df:
    try:
        country_code = geocoder.ip(ip_address).country
        ip_country_codes.append(country_code)
        counter += 1
    except Exception as e:
        error_str = str(e)
        # Print the exact error for debugging
        print(f"ERROR CAUGHT: {error_str}")

        # Check for rate limit indicators in various ways
        if any(text in error_str.lower() for text in ["429", "too many requests", "rate limit"]):
            print('Rate limit detected! Sleeping for a minute...')
            time.sleep(60)

            # Try once more with the same IP
            try:
                country_code = geocoder.ip(ip_address).country
                ip_country_codes.append(country_code)
                counter += 1
            except:
                print(f"Failed on retry for {ip_address}, adding None")
                ip_country_codes.append(None)
        else:
            print(f"Other error for {ip_address}, adding None")
            ip_country_codes.append(None)

ERROR:geocoder.base:Status code 429 from http://ipinfo.io/162.198.218.92/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/162.198.218.92/json
ERROR:geocoder.base:Status code 429 from http://ipinfo.io/13.149.61.4/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/13.149.61.4/json
ERROR:geocoder.base:Status code 429 from http://ipinfo.io/215.97.143.157/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/215.97.143.157/json
ERROR:geocoder.base:Status code 429 from http://ipinfo.io/200.13.225.150/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/200.13.225.150/json
ERROR:geocoder.base:Status code 429 from http://ipinfo.io/65.164.3.100/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/65.164.3.100/json
ERROR:geocoder.base:Status code 429 from http://ipinfo.io/117.67.192.211/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/117.67.192.211/j

KeyboardInterrupt: 

In [38]:
try:
  f = pd.read_csv('tite.csv')
except Exception as e:
  print('Walang file')

Walang file


In [31]:
ip_country_codes

[None, None, None, None, None, None, None, None, None, None]

In [ ]:
ip_address_df

###### Location column

In [39]:
# Initialize Nominatim API
geolocator = Nominatim(user_agent="placefinder")

In [ ]:
# locations = data['Location']
# loc_country_codes = []
# loc_counter = 0

# for location in locations:
#   try:
#     country_code = geolocator.geocode(query=location, addressdetails=True).raw.get('address', {}).get('country_code').upper()
#     loc_country_codes.append(country_code)
#     loc_counter += 1
#   except Exception as e:
#     error_str = str(e)
#     if any(text in error_str for text in ['Retrying', 'timeout', 'Too Many Requests']):
#       print('Sleeping...')
#       time.sleep(60)

###### Combining the two locations

In [85]:
# location_df = data[['Location', 'IP Address']]
# location_df['Location CC'] = loc_country_codes
# location_df['IP CC'] = ip_country_codes
# location_df

<ipython-input-85-da3021ef1b07>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  location_df['Location CC'] = loc_country_codes
<ipython-input-85-da3021ef1b07>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  location_df['IP CC'] = ip_country_codes


,Location,IP Address,Location CC,IP CC
0,San Diego,162.198.218.92,US,US
1,Houston,13.149.61.4,US,US
2,Mesa,215.97.143.157,US,US
3,Raleigh,200.13.225.150,US,CO
4,Atlanta,65.164.3.100,US,CA
...,...,...,...,...
2507,Colorado Springs,21.157.41.17,US,US
2508,Tucson,49.174.157.140,US,KR
2509,San Diego,58.1.27.124,US,JP
2510,Denver,21.190.11.223,US,US


In [87]:
# location_df.to_csv('/content/drive/MyDrive/Colab Notebooks/Term deposit availing prediction/Data/locations.csv', index=False)

In [89]:
location_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Term deposit availing prediction/Data/locations.csv')
location_df['isEqual'] = location_df['Location CC'] == location_df['IP CC']
location_df

,Location,IP Address,Location CC,IP CC,isEqual
0,San Diego,162.198.218.92,US,US,True
1,Houston,13.149.61.4,US,US,True
2,Mesa,215.97.143.157,US,US,True
3,Raleigh,200.13.225.150,US,CO,False
4,Atlanta,65.164.3.100,US,CA,False
...,...,...,...,...,...
2507,Colorado Springs,21.157.41.17,US,US,True
2508,Tucson,49.174.157.140,US,KR,False
2509,San Diego,58.1.27.124,US,JP,False
2510,Denver,21.190.11.223,US,US,True


Let's put `location_df['isEqual']` into `card_column`.

In [92]:
cat_column['isLocationEqual'] = location_df['isEqual']
cat_column

<ipython-input-92-3972a15a420b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cat_column['isLocationEqual'] = location_df['isEqual']


,TransactionType,Channel,CustomerOccupation,isLocationEqual
0,Debit,ATM,Doctor,True
1,Debit,ATM,Doctor,True
2,Debit,Online,Student,True
3,Debit,Online,Student,False
4,Credit,Online,Student,False
...,...,...,...,...
2507,Credit,Branch,Doctor,True
2508,Debit,Branch,Doctor,False
2509,Debit,Branch,Retired,False
2510,Debit,Online,Student,True


#### ColumnTransformer

In [98]:
preprocessor = ColumnTransformer(
    [
        ('categorical', card_processor, list(cat_column.columns)),
        ('numerical', num_processor, list(num_column.columns))
    ]
)

In [99]:
preprocessor

ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='missing',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['TransactionType', 'Channel',
                                  'CustomerOccupation', 'isLocationEqual']),
                                ('numerical',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', MinMaxScaler())]),
                                 ['TransactionAmount', 'CustomerAge',
                                  'TransactionDuration', 'LoginAttempts',
                                  'AccountBalance', 'interval'])])

In [101]:
data_new = pd.concat([num_column, cat_column], axis=1)
data_new

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,interval,TransactionType,Channel,CustomerOccupation,isLocationEqual
0,14.09,70,81,1,5112.21,49477134.0,Debit,ATM,Doctor,True
1,376.24,68,141,1,13758.91,42823516.0,Debit,ATM,Doctor,True
2,126.29,19,56,1,1122.35,41694656.0,Debit,Online,Student,True
3,184.50,26,25,1,8569.06,47403415.0,Debit,Online,Student,False
4,13.45,26,198,1,7429.40,33228915.0,Credit,Online,Student,False
...,...,...,...,...,...,...,...,...,...,...
2507,856.21,33,109,1,12690.79,48178913.0,Credit,Branch,Doctor,True
2508,251.54,48,177,1,254.75,51201294.0,Debit,Branch,Doctor,False
2509,28.63,56,146,1,3382.91,38069989.0,Debit,Branch,Retired,False
2510,185.97,23,19,1,1776.91,53452056.0,Debit,Online,Student,True


Now, we can run `data_new` on `preprocessor`. However, when making the dataframe, we can't just use the columns of `data_new` since the one hot encoder split the columns into their unique values. For the columns, we will use `preprocessor.get_feature_names_out()`.

In [107]:
arr_trans = preprocessor.fit_transform(data_new)
data_trans = pd.DataFrame(arr_trans, columns=preprocessor.get_feature_names_out())
data_trans

,categorical__TransactionType_Credit,categorical__TransactionType_Debit,categorical__Channel_ATM,categorical__Channel_Branch,categorical__Channel_Online,categorical__CustomerOccupation_Doctor,categorical__CustomerOccupation_Engineer,categorical__CustomerOccupation_Retired,categorical__CustomerOccupation_Student,categorical__isLocationEqual_False,categorical__isLocationEqual_True,numerical__TransactionAmount,numerical__CustomerAge,numerical__TransactionDuration,numerical__LoginAttempts,numerical__AccountBalance,numerical__interval
0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.007207,0.838710,0.244828,0.0,0.336832,0.728036
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.195940,0.806452,0.451724,0.0,0.918055,0.516531
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.065680,0.016129,0.158621,0.0,0.068637,0.480647
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.096016,0.129032,0.051724,0.0,0.569198,0.662117
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.006874,0.129032,0.648276,0.0,0.492591,0.211537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2507,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.446074,0.241935,0.341379,0.0,0.846257,0.686768
2508,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.130953,0.483871,0.575862,0.0,0.010318,0.782844
2509,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.014785,0.612903,0.468966,0.0,0.220590,0.365426
2510,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.096782,0.080645,0.031034,0.0,0.112636,0.854391


### Splitting the data

Since both of our methods involve unsupervised learning, we do not have to split the data anymore to train, test, and validation.

## Modeling data

As aforementioned, we will use `AutoEncoder` and `DBSCAN` to look for anomalies.

### AutoEncoder

We will use the model from [this website](https://medium.com/@weidagang/demystifying-anomaly-detection-with-autoencoder-neural-networks-1e235840d879).

In [115]:
# Get number of columns
amt_columns = data_trans.shape[1]
amt_columns

17

In [120]:
# Build the AutoEncoder model
model_auto = keras.Sequential([
    # Encoder: Reduce dimensionality, learn the most important features
    keras.layers.Dense(128, activation='relu', input_shape=(amt_columns,)), # Reducing dimension to 128
    keras.layers.Dense(64, activation='relu'), # Further reducing dimension to 64
    keras.layers.Dense(32, activation='relu'), # Further reducing to the most compact form (bottleneck layer)

    # Decoder: Reconstruct the image from the reduced representation
    keras.layers.Dense(64, activation='relu'), # Start expanding dimension
    keras.layers.Dense(128, activation='relu'), # Continue expanding dimension
    keras.layers.Dense(amt_columns, activation='sigmoid') # Restore to original image size
])

# Compile model
model_auto.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [122]:
model_auto.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 128)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 17)             │         2,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,797 (296.09 KB)

 Trainable params: 25,265 (98.69 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 50,532 (197.39 KB)

Then, we can train the model.

In [121]:
history_auto = model_auto.fit(data_trans, data_trans, epochs=10, batch_size=100)

Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1868
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1205
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0526
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0170
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0107
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0086
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0069
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0051
Epoch 9/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0042
Epoch 10/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0043


In [125]:
pd.DataFrame(history_auto.history)

,loss
0,0.170138
1,0.106368
2,0.040379
3,0.015073
4,0.010514
5,0.008092
6,0.006185
7,0.004872
8,0.004342
9,0.004077


### DBSCAN

Let's make an elbow plot using DBSCAN.

In [127]:
model_db = DBSCAN()
model_db.fit_predict(data_trans)

array([ 0,  0,  1, ..., 11,  1,  2])

In [128]:
model_db.get_params()

{'algorithm': 'auto',
 'eps': 0.5,
 'leaf_size': 30,
 'metric': 'euclidean',
 'metric_params': None,
 'min_samples': 5,
 'n_jobs': None,
 'p': None}

In [129]:
data_final = data_trans.copy()
data_final['cluster'] = model_db.labels_
data_final

,categorical__TransactionType_Credit,categorical__TransactionType_Debit,categorical__Channel_ATM,categorical__Channel_Branch,categorical__Channel_Online,categorical__CustomerOccupation_Doctor,categorical__CustomerOccupation_Engineer,categorical__CustomerOccupation_Retired,categorical__CustomerOccupation_Student,categorical__isLocationEqual_False,categorical__isLocationEqual_True,numerical__TransactionAmount,numerical__CustomerAge,numerical__TransactionDuration,numerical__LoginAttempts,numerical__AccountBalance,numerical__interval,cluster
0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.007207,0.838710,0.244828,0.0,0.336832,0.728036,0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.195940,0.806452,0.451724,0.0,0.918055,0.516531,0
2,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.065680,0.016129,0.158621,0.0,0.068637,0.480647,1
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.096016,0.129032,0.051724,0.0,0.569198,0.662117,6
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.006874,0.129032,0.648276,0.0,0.492591,0.211537,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2507,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.446074,0.241935,0.341379,0.0,0.846257,0.686768,40
2508,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.130953,0.483871,0.575862,0.0,0.010318,0.782844,12
2509,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.014785,0.612903,0.468966,0.0,0.220590,0.365426,11
2510,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.096782,0.080645,0.031034,0.0,0.112636,0.854391,1


In [133]:
import plotly.express as px

fig = px.scatter(data_final, x='numerical__TransactionAmount', y='numerical__interval', color='cluster')
fig.show()

In [126]:
clusters, inertias = [], []
max_k = 10

for k in range(1, max_k):
  # Fitting the data
  model_db = DBSCAN(n_cluster)
  model_db.fit(data_trans)

  # Appending means and inertia
  clusters.append(k)
  inertias.append(model_db.inertia_)

AttributeError: 'DBSCAN' object has no attribute 'inertia_'